# One-dimensional axial-radius comparison

Translate the MicroBooNE dipole and $z$-expansion extractions, the deuterium dipole and $z$-expansion results, and the MINERvA and LQCD $z$-expansion results into the common observable

$$r_A^2=-\frac{6}{g_A}\left.\frac{dF_A}{dQ^2}\right|_{Q^2=0}.$$

The transformation is applied sample by sample, retaining coefficient correlations and non-Gaussian posterior shapes. Values are reported in $\mathrm{fm}^2$. The intervals below are central equal-tailed 68% (approximately one-standard-deviation) credible/confidence intervals.

Four MicroBooNE posteriors are shown, in two groups. The uniform-prior fits (`ma_uniform`, `minerva_k6_uniform`) put no Gaussian pull penalty on the axial parameters, so they carry the constraining power of the data alone; the external-prior fits (`minerva_k6`, `lqcd_k6`) update a published prior, so each should be read against the external row of the same colour further down. The two groups are set apart by their own background tint in the forest plots, and the external determinations sit on the plain background.


In [ ]:
from pathlib import Path
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate this repository whether the notebook is launched here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if helper_dir is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FA_SOURCE_COLORS, PUBLICATION_RC, SPECS, _reference_prior_samples,
    column_font_sizes, load_fit,
)

HBARC_GEV_FM = 0.1973269804
SUITES = (
    ('nuwro_fit_results', 'NuWro'),
    ('asimov_fit_results', 'Asimov'),
    ('opendata_fit_results', 'Open data'),
)
# MicroBooNE posteriors, in two groups: the fits whose axial parameters carry
# no Gaussian pull penalty, and the fits that update an external prior. Each
# group gets its own background tint in the forest plots; the external
# determinations below them get none. Entries are (fit key, row label), and the
# row order here is the row order of every figure and table in this notebook.
UNIFORM_PRIOR_FITS = (
    ('ma_uniform', r'Posterior from uniform $M_A$ prior'),
    ('minerva_k6_uniform', r'Posterior from uniform prior, $k_{\max}=6$'),
)
EXTERNAL_PRIOR_FITS = (
    ('minerva_k6', r'Posterior from MINERvA prior, $k_{\max}=6$'),
    ('lqcd_k6', r'Posterior from LQCD prior, $k_{\max}=6$'),
)
# The r_A^2 >= 0 diagnostic applies to this chain alone: it is the only one
# broad enough to reach negative radii.
MICROBOONE_ZEXP_FIT = 'minerva_k6_uniform'
REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2 = False  # diagnostic truncation only
N_REFERENCE_SAMPLES = 200_000
RANDOM_SEED = 2026
CONFIDENCE_LEVELS = (0.68,)


mpl.rcParams.update(PUBLICATION_RC)


## Transformation

For a dipole, $F_A(Q^2)=g_A(1+Q^2/M_A^2)^{-2}$, so $r_A^2=12/M_A^2$. For $F_A=\sum_k a_k z^k$, the derivative is evaluated analytically using the basis metadata belonging to each result. The factor $(\hbar c)^2$ converts $\mathrm{GeV}^{-2}$ to $\mathrm{fm}^2$.

In [ ]:
def radius_squared_from_ma(ma_gev):
    ma = np.asarray(ma_gev, dtype=float)
    if np.any(~np.isfinite(ma)) or np.any(ma <= 0):
        raise ValueError('Every M_A sample must be finite and positive')
    return 12.0 / ma**2 * HBARC_GEV_FM**2


def equivalent_ma_from_radius_squared(radius_squared_fm2):
    """Dipole M_A with the same Q2=0 slope; defined only for r_A^2 > 0."""
    radius_squared = np.asarray(radius_squared_fm2, dtype=float)
    if np.any(~np.isfinite(radius_squared)) or np.any(radius_squared <= 0):
        raise ValueError('Every r_A^2 sample must be finite and positive')
    return np.sqrt(12.0 * HBARC_GEV_FM**2 / radius_squared)


def dz_dq2_at_zero(t0_gev2, t_cut_gev2):
    # z=(sqrt(tcut+Q2)-sqrt(tcut-t0))/(sqrt(tcut+Q2)+sqrt(tcut-t0))
    a = np.sqrt(t_cut_gev2)
    b = np.sqrt(t_cut_gev2 - t0_gev2)
    return b / (a * (a + b)**2)


def radius_squared_from_zexp(coefficients, t0_gev2, t_cut_gev2, g_a):
    coefficients = np.atleast_2d(np.asarray(coefficients, dtype=float))
    z0 = ((np.sqrt(t_cut_gev2) - np.sqrt(t_cut_gev2 - t0_gev2)) /
          (np.sqrt(t_cut_gev2) + np.sqrt(t_cut_gev2 - t0_gev2)))
    k = np.arange(1, coefficients.shape[1])
    dfa_dq2 = (coefficients[:, 1:] @ (k * z0**(k - 1))) * dz_dq2_at_zero(
        t0_gev2, t_cut_gev2
    )
    return -6.0 / g_a * dfa_dq2 * HBARC_GEV_FM**2


def transform_a1_to_t0_zero(coefficients, t0_gev2, t_cut_gev2):
    """Return the exact coefficient of z_new about Q2=0 (t0_new=0)."""
    coefficients = np.atleast_2d(np.asarray(coefficients, dtype=float))
    # z_old = (z_new + c)/(1 + c*z_new), with c=z_old(Q2=0).
    c = ((np.sqrt(t_cut_gev2) - np.sqrt(t_cut_gev2 - t0_gev2)) /
         (np.sqrt(t_cut_gev2) + np.sqrt(t_cut_gev2 - t0_gev2)))
    k = np.arange(1, coefficients.shape[1])
    return (1.0 - c**2) * (coefficients[:, 1:] @ (k * c**(k - 1)))


def radius_squared_from_t0_zero_a1(coefficients, t0_gev2, t_cut_gev2, g_a):
    a1_t0_zero = transform_a1_to_t0_zero(coefficients, t0_gev2, t_cut_gev2)
    # For t0=0, z(0)=0 and dz/dQ2|0 = 1/(4*t_cut).
    return -3.0 * a1_t0_zero / (2.0 * g_a * t_cut_gev2) * HBARC_GEV_FM**2


def central_interval(samples, probability):
    """Equal-tailed interval and median.

    ``method="hazen"`` puts the k-th of n order statistics at (k - 0.5) / n, the
    convention these intervals have always used.
    """
    tail = (1.0 - probability) / 2.0
    return np.quantile(samples, [tail, 0.5, 1.0 - tail], axis=0, method="hazen")


# Numerical cross-checks of both analytic transformations.
assert np.isclose(radius_squared_from_ma([1.0])[0] / HBARC_GEV_FM**2, 12.0)
test_coefficients, test_t0, test_tcut = np.array([[1.0, -2.0, 0.5]]), -0.5, 0.2
eps = 1e-7
def test_fa(q2):
    z = ((np.sqrt(test_tcut + q2) - np.sqrt(test_tcut - test_t0)) /
         (np.sqrt(test_tcut + q2) + np.sqrt(test_tcut - test_t0)))
    return np.sum(test_coefficients[0] * z**np.arange(test_coefficients.shape[1]))
numeric_slope = (test_fa(eps) - test_fa(0.0)) / eps
analytic_r2 = radius_squared_from_zexp(test_coefficients, test_t0, test_tcut, -1.27)[0]
basis_r2 = radius_squared_from_t0_zero_a1(
    test_coefficients, test_t0, test_tcut, -1.27
)[0]
assert np.isclose(analytic_r2, -6 / -1.27 * numeric_slope * HBARC_GEV_FM**2, rtol=2e-6)
assert np.isclose(analytic_r2, basis_r2, rtol=2e-14, atol=2e-14)

## Build the one-dimensional distributions

`UNIFORM_PRIOR_FITS` and `EXTERNAL_PRIOR_FITS` above select the MicroBooNE rows: the first pair has no Gaussian pull penalty on its axial parameters, the second updates a published prior. Add or remove entries there to compare different fit variants, and keep the matching block size in `ROW_GROUPS` in step. Setting `REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2=True` discards MicroBooNE $z$-expansion MCMC samples with $r_A^2<0$. This is a diagnostic conditional distribution, not a refit or a physical prior, and the retained fraction is reported explicitly. The deuterium dipole input is $M_A=1.014\pm0.014$ GeV. Every $z$-expansion result shown uses the common $k_{\max}=6$, $t_0=-0.50\ \mathrm{GeV}^2$, $t_{\mathrm{cut}}=9(0.134\ \mathrm{GeV})^2$, and $F_A(0)=-1.2754$ convention. The deuterium input is the published result translated to this common basis; MINERvA and LQCD use their correlated $k_{\max}=6$ coefficient covariance matrices.

In [ ]:
def build_suite_results(suite):
    spec_by_key = {spec.key: spec for spec in SPECS}
    posterior_fits = (*UNIFORM_PRIOR_FITS, *EXTERNAL_PRIOR_FITS)
    fits = {}
    for key, _ in posterior_fits:
        result = load_fit(spec_by_key[key], suite)
        if result is None:
            raise FileNotFoundError(f'Missing fit output for {key!r} in {suite}')
        fits[key] = result
        print(f'{key:20s} | {len(result["samples"]):,} posterior samples')

    rng = np.random.default_rng(RANDOM_SEED)
    deuterium_ma = rng.normal(1.014, 0.014, N_REFERENCE_SAMPLES)
    deuterium_coeff, deuterium_t0, deuterium_tcut = _reference_prior_samples(
        'deuterium_k6', N_REFERENCE_SAMPLES, RANDOM_SEED + 1
    )
    minerva_coeff, minerva_t0, minerva_tcut = _reference_prior_samples(
        'minerva_k6', N_REFERENCE_SAMPLES, RANDOM_SEED + 2
    )
    lqcd_coeff, lqcd_t0, lqcd_tcut = _reference_prior_samples(
        'lqcd_k6', N_REFERENCE_SAMPLES, RANDOM_SEED + 3
    )

    # The MicroBooNE posteriors, in the order the forest-plot row groups
    # expect: the uniform-prior fits, then the external-prior fits.
    # `zexp_inputs` collects every z-expansion distribution for the
    # independent t0=0 basis check further down.
    distributions = {}
    zexp_inputs = {}
    diagnostic_status = None
    for key, label in posterior_fits:
        result = fits[key]
        spec = result['spec']
        if spec.prior is None:
            # Dipole M_A fit: r_A^2 = 12/M_A^2, with no z-expansion basis.
            distributions[label] = radius_squared_from_ma(result['samples'][:, 0])
            continue
        prior = spec.prior
        coefficients = result['samples']
        radius_squared = radius_squared_from_zexp(
            coefficients, prior.t0_gev2, prior.t_cut_gev2, prior.fa_q2_zero
        )
        if key == MICROBOONE_ZEXP_FIT:
            positive_r2_mask = radius_squared >= 0.0
            diagnostic_status = pd.DataFrame({
                'positive-r_A^2 requirement enabled': [REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2],
                'total MicroBooNE z-expansion samples': [len(radius_squared)],
                'retained samples': [int(positive_r2_mask.sum())
                                     if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2
                                     else len(radius_squared)],
                'positive-r_A^2 fraction': [positive_r2_mask.mean()],
            })
            if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2:
                if not positive_r2_mask.any():
                    raise ValueError(
                        'No MicroBooNE z-expansion samples have non-negative r_A^2'
                    )
                coefficients = coefficients[positive_r2_mask]
                radius_squared = radius_squared[positive_r2_mask]
                label += r' ($r_A^2\geq0$ diagnostic)'
        distributions[label] = radius_squared
        zexp_inputs[label] = (coefficients, prior.t0_gev2, prior.t_cut_gev2,
                              prior.fa_q2_zero)
    if diagnostic_status is not None:
        display(diagnostic_status.style.format({
            'positive-r_A^2 fraction': '{:.3%}',
        }))

    # External determinations: unweighted draws from the published results. The
    # dipole comes first so the rows line up with `INTERVAL_COLORS`.
    external_zexp = {
        r'Deuterium (2025) prior, $k_{\max}=6$': (
            deuterium_coeff, deuterium_t0, deuterium_tcut, -1.2754
        ),
        r'MINERvA (2026) prior, $k_{\max}=6$': (
            minerva_coeff, minerva_t0, minerva_tcut, -1.2754
        ),
        r'LQCD (2026) prior, $k_{\max}=6$': (
            lqcd_coeff, lqcd_t0, lqcd_tcut, -1.2754
        ),
    }
    distributions[r'Deuterium dipole, $M_A=1.014\pm0.014$ GeV'] = (
        radius_squared_from_ma(deuterium_ma)
    )
    for label, (coefficients, t0, t_cut, g_a) in external_zexp.items():
        distributions[label] = radius_squared_from_zexp(
            coefficients, t0, t_cut, g_a
        )
    zexp_inputs.update(external_zexp)

    # Independent double-check: transform to the t0=0 basis, where the radius
    # depends on a1 alone. Agreement is tested for every individual sample.
    basis_check_records = []
    for label, (coefficients, t0, tcut, g_a) in zexp_inputs.items():
        direct = distributions[label]
        transformed = radius_squared_from_t0_zero_a1(coefficients, t0, tcut, g_a)
        difference = transformed - direct
        basis_check_records.append({
            'result': label,
            'median transformed a1 (t0=0)': np.median(
                transform_a1_to_t0_zero(coefficients, t0, tcut)
            ),
            'median r_A^2 direct': np.median(direct),
            'median r_A^2 via t0=0': np.median(transformed),
            'max abs sample difference': np.max(np.abs(difference)),
        })
        assert np.allclose(transformed, direct, rtol=2e-13, atol=2e-13)
    basis_check = pd.DataFrame(basis_check_records).set_index('result')
    display(basis_check.style.format('{:.12g}'))

    records = []
    for label, samples in distributions.items():
        mean, std = samples.mean(axis=0), samples.std(axis=0)
        record = {'result': label, 'mean': mean, 'std': std}
        for probability in CONFIDENCE_LEVELS:
            low, median, high = central_interval(samples, probability)
            record.update({
                'median': median, f'low_{probability:.2f}': low, f'high_{probability:.2f}': high
            })
        records.append(record)
    summary = pd.DataFrame(records).set_index('result')
    display(summary.style.format('{:.4f}'))

    equivalent_ma_distributions = {}
    equivalent_ma_records = []
    for label, radius_samples in distributions.items():
        positive = radius_samples > 0.0
        if not positive.any():
            raise ValueError(f'{label} has no positive r_A^2 samples to map to M_A')
        ma_samples = equivalent_ma_from_radius_squared(radius_samples[positive])
        equivalent_ma_distributions[label] = ma_samples
        low, median, high = central_interval(ma_samples, 0.68)
        mean, std = ma_samples.mean(axis=0), ma_samples.std(axis=0)
        equivalent_ma_records.append({
            'result': label,
            'positive-r_A^2 fraction': positive.mean(),
            'retained samples': positive.sum(),
            'total samples': len(radius_samples),
            'mean M_A [GeV]': mean,
            'std M_A [GeV]': std,
            'median': median, 'low_0.68': low, 'high_0.68': high,
        })
    equivalent_ma_summary = pd.DataFrame(equivalent_ma_records).set_index('result')
    display(equivalent_ma_summary.style.format({
        'positive-r_A^2 fraction': '{:.3%}',
        'mean M_A [GeV]': '{:.4f}', 'std M_A [GeV]': '{:.4f}',
        'median': '{:.4f}', 'low_0.68': '{:.4f}', 'high_0.68': '{:.4f}',
    }))
    return {
        'distributions': distributions,
        'summary': summary,
        'diagnostic_status': diagnostic_status,
        'basis_check': basis_check,
        'equivalent_ma_distributions': equivalent_ma_distributions,
        'equivalent_ma_summary': equivalent_ma_summary,
    }


## Common comparison

The first forest and density plots compare $r_A^2$. The second pair applies the inverse dipole relation $M_A^{\mathrm{equiv}}=\sqrt{12(\hbar c)^2/r_A^2}$, giving the dipole mass with the same slope at $Q^2=0$. This inverse is real only for $r_A^2>0$, so non-positive samples are excluded and the retained fraction is reported explicitly. All forest plots show central 68% intervals, with filled circles for the MicroBooNE posteriors and hollow squares for the external inputs. Each row is named on the $y$ axis in its own colour, so the forest plots carry no legend. Dashed rules and background tints separate the three blocks: the posteriors from uniform priors in the top band, the posteriors from external priors in the second, and the external determinations on the plain background. A posterior from an external prior keeps the colour of the prior it updates, so the two rows can be read as a pair; in the density panels the posterior is the solid curve and the external input the broken one. Density panels are normalized to unit area.

In [ ]:
INTERVAL_COLORS = [
    # Posteriors from uniform priors.
    FA_SOURCE_COLORS['ma'], FA_SOURCE_COLORS['minerva_k6_uniform'],
    # Posteriors from external priors: each keeps the colour of the prior it
    # updates, so the two can be read as a pair across the dashed rule. In the
    # density panels the posterior is solid and the prior broken.
    FA_SOURCE_COLORS['minerva_k6'], FA_SOURCE_COLORS['lqcd_k6'],
    # External determinations.
    '#4D4D4D', FA_SOURCE_COLORS['deuterium'], FA_SOURCE_COLORS['minerva_k6'],
    FA_SOURCE_COLORS['lqcd_k6'],
]
# Consecutive blocks of rows, top to bottom, matching the order `distributions`
# is built in. The two posterior blocks each get their own background tint, a
# filled circle and a solid density curve; the external determinations get no
# tint, a hollow square and a broken curve. `size` must add up to the number of
# rows, so a fit added to either list above belongs in the matching block here.
ROW_GROUPS = (
    dict(size=len(UNIFORM_PRIOR_FITS), band=FA_SOURCE_COLORS['ma'],
         marker='o', filled=True, dash='-'),
    dict(size=len(EXTERNAL_PRIOR_FITS), band='#E69F00',
         marker='o', filled=True, dash='-'),
    # Deuterium dipole plus the three k_max=6 external priors.
    dict(size=4, band=None, marker='s', filled=False, dash=(0, (4.5, 2.4))),
)
# One style per row, expanded from the blocks above.
ROW_STYLES = [group for group in ROW_GROUPS for _ in range(group['size'])]
BAND_ALPHA = 0.055
# Row labels carry their series colour, so they are darkened until the text
# reads against white; the light sky blue is the entry that needs it.
LABEL_MAX_LUMINANCE = 0.30
GRID_COLOR = '#9AA4B2'
RULE_COLOR = '#C3CAD4'
NOTE_COLOR = '#596273'
# Both figures below and the corner plots are placed at one column width in the
# paper, so these wider source figures are shrunk further by LaTeX. Their fonts
# are scaled by the width ratio so that everything prints at the same text size.
# The forest plot keeps a fixed width, so its fonts do not move when rows are
# added; only its height grows.
INTERVAL_WIDTH = 8.2
INTERVAL_ROW_HEIGHT = 0.70
INTERVAL_FIXED_HEIGHT = 0.90
DENSITY_FIGSIZE = (7.2, 6.0)
DENSITY_FONT_SIZES = column_font_sizes(DENSITY_FIGSIZE[0])
# The legend sits inside the density axes; at the scaled-up font size its eight
# entries need this much of the height left clear above the tallest curve.
LEGEND_HEADROOM = 2.0


def interval_figsize(n_rows):
    """Forest-plot size: fixed width, height growing with the row count."""
    return (INTERVAL_WIDTH,
            INTERVAL_FIXED_HEIGHT + INTERVAL_ROW_HEIGHT * n_rows)


# Display names for the forest-plot rows, keyed by the label each result
# carries in `distributions` and in the summary index. Two short lines keep the
# label column narrower than the data: the source on the first line, the
# parameter that identifies the result on the second. A label with no entry
# here falls through to the y axis unwrapped.
ROW_LABEL_WRAPS = {
    r'Posterior from uniform $M_A$ prior':
        'Posterior\n' r'Uniform $M_A$ prior',
    r'Posterior from uniform prior, $k_{\max}=6$':
        'Posterior\n' r'Uniform $k_{\max}=6$ prior',
    r'Posterior from uniform prior, $k_{\max}=6$ ($r_A^2\geq0$ diagnostic)':
        'Posterior\n' r'Uniform $k_{\max}=6$ prior' '\n' r'($r_A^2\geq0$ diagnostic)',
    r'Posterior from MINERvA prior, $k_{\max}=6$':
        'Posterior\n' r'MINERvA $k_{\max}=6$ prior',
    r'Posterior from LQCD prior, $k_{\max}=6$':
        'Posterior\n' r'LQCD $k_{\max}=6$ prior',
    r'Deuterium dipole, $M_A=1.014\pm0.014$ GeV':
        'Deuterium dipole\n' r'$M_A=1.014\pm0.014$ GeV',
    r'Deuterium (2025) prior, $k_{\max}=6$':
        'Deuterium prior\n' r'$k_{\max}=6$',
    r'MINERvA (2026) prior, $k_{\max}=6$':
        'MINERvA prior\n' r'$k_{\max}=6$',
    r'LQCD (2026) prior, $k_{\max}=6$':
        'LQCD prior\n' r'$k_{\max}=6$',
}


def label_color(color):
    """Series colour darkened to a legible luminance for row-label text."""
    rgb = np.array(mpl.colors.to_rgb(color))
    luminance = float(np.dot(rgb, (0.2126, 0.7152, 0.0722)))
    if luminance <= LABEL_MAX_LUMINANCE:
        return tuple(rgb)
    return tuple(rgb * (LABEL_MAX_LUMINANCE / luminance))


def plot_interval_summary(summary, labels, xlabel, colors=INTERVAL_COLORS,
                          groups=ROW_GROUPS, figsize=None):
    """Forest plot of central 68% intervals, one named row per result.

    Rows run top to bottom in the order of `labels` and are named on the y
    axis in their own series colour, so the figure needs no legend and every
    interval is identified where it is drawn. `ROW_LABEL_WRAPS` supplies the
    short two-line name each row is drawn with. `groups` splits the rows into
    consecutive blocks separated by dashed rules: the MicroBooNE posteriors
    from uniform priors and those from external priors each sit in a tinted
    band of their own and keep filled circles, while the external
    determinations below keep hollow squares on the plain background. Each
    interval is a soft wide band under a thin bar with end caps, and the marker
    is the median. The x range covers only the data, with a small symmetric
    margin.
    """
    if sum(group['size'] for group in groups) != len(labels):
        raise ValueError(
            f'The row groups cover {sum(g["size"] for g in groups)} rows, '
            f'but {len(labels)} were given'
        )
    figsize = figsize or interval_figsize(len(labels))
    font_sizes = column_font_sizes(figsize[0])
    y = np.arange(len(labels))[::-1]
    # First row of each block, so a band and its rule follow the block's own
    # extent instead of a hard-coded row number.
    starts = np.cumsum([0] + [group['size'] for group in groups[:-1]])
    styles = [group for group in groups for _ in range(group['size'])]

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    for start in starts[1:]:
        ax.axhline(y[start] + 0.5, color=RULE_COLOR, linewidth=0.9,
                   linestyle=(0, (4, 3.5)), zorder=1)

    for row_index, (label, color) in enumerate(zip(labels, colors)):
        row = summary.loc[label]
        style = styles[row_index]
        low, median, high = row['low_0.68'], row['median'], row['high_0.68']
        row_y = y[row_index]
        ax.plot([low, high], [row_y, row_y], color=color, linewidth=7.0,
                alpha=0.18, solid_capstyle='butt', zorder=2)
        ax.plot([low, high], [row_y, row_y], color=color, linewidth=1.8,
                solid_capstyle='butt', zorder=3)
        for edge in (low, high):
            ax.plot([edge, edge], [row_y - 0.14, row_y + 0.14], color=color,
                    linewidth=1.8, solid_capstyle='butt', zorder=3)
        ax.plot(
            [median], [row_y], marker=style['marker'],
            color=color, markerfacecolor=color if style['filled'] else 'white',
            markeredgecolor=color, markeredgewidth=1.8,
            markersize=8.5 if style['filled'] else 7.4, zorder=4, clip_on=False,
        )

    ax.set_yticks(y)
    ax.set_yticklabels([ROW_LABEL_WRAPS.get(label, label) for label in labels])
    for row_index, (tick_label, color) in enumerate(
        zip(ax.get_yticklabels(), colors)
    ):
        tick_label.set_color(label_color(color))
        tick_label.set_fontsize(font_sizes['tick'])
        tick_label.set_linespacing(1.3)
        if styles[row_index]['filled']:
            tick_label.set_fontweight('semibold')
    ax.tick_params(axis='y', which='both', left=False, right=False, pad=9)
    ax.tick_params(axis='x', which='major', labelsize=font_sizes['tick'])
    ax.set_xlabel(xlabel, labelpad=7, fontsize=font_sizes['label'])
    ax.grid(axis='x', which='major', color=GRID_COLOR, alpha=0.22,
            linewidth=0.7, zorder=0)
    ax.minorticks_on()
    ax.annotate(
        'Central 68% intervals', xy=(1.0, 1.015), xycoords='axes fraction',
        ha='right', va='bottom', color=NOTE_COLOR, fontsize=font_sizes['legend'],
    )

    low = summary.loc[labels, 'low_0.68'].min()
    high = summary.loc[labels, 'high_0.68'].max()
    span = high - low
    ax.set_xlim(low - 0.07 * span, high + 0.07 * span)
    ax.set_ylim(-0.72, len(labels) - 0.28)
    # Tint each posterior block. Drawn after the y limits are fixed so the top
    # band ends flush with the frame.
    top = ax.get_ylim()[1]
    for group, start in zip(groups, starts):
        if group['band'] is None:
            continue
        ax.axhspan(y[start + group['size'] - 1] - 0.5,
                   top if start == 0 else y[start] + 0.5,
                   color=group['band'], alpha=BAND_ALPHA, linewidth=0, zorder=0)
    return fig


def plot_suite_results(suite, distributions, summary):
    labels = list(distributions)
    interval_fig = plot_interval_summary(
        summary, labels, r'$r_A^2\ [\mathrm{fm}^2]$'
    )

    all_low = min(np.quantile(values, 0.001) for values in distributions.values())
    all_high = max(np.quantile(values, 0.999) for values in distributions.values())
    density_fig, ax_density = plt.subplots(
        figsize=DENSITY_FIGSIZE, constrained_layout=True
    )
    peak = 0.0
    for (label, samples), color, style in zip(
        distributions.items(), INTERVAL_COLORS, ROW_STYLES
    ):
        counts, edges = np.histogram(samples, bins=180, range=(all_low, all_high),
                                     density=True)
        centers = (edges[1:] + edges[:-1]) / 2
        kernel_x = np.arange(-8, 9)
        kernel = np.exp(-0.5 * (kernel_x / 2.0)**2); kernel /= kernel.sum()
        density = np.convolve(counts, kernel, mode='same')
        peak = max(peak, density.max())
        ax_density.plot(centers, density, color=color, linewidth=2, label=label,
                        linestyle=style['dash'])

    # Headroom for the legend, which sits inside the axes.
    ax_density.set_ylim(0, LEGEND_HEADROOM * peak)
    ax_density.set_xlabel(r'$r_A^2\ [\mathrm{fm}^2]$',
                          fontsize=DENSITY_FONT_SIZES['label'])
    ax_density.set_ylabel('Probability density',
                          fontsize=DENSITY_FONT_SIZES['label'])
    ax_density.tick_params(labelsize=DENSITY_FONT_SIZES['tick'])
    ax_density.grid(color='#9AA4B2', alpha=0.18, linewidth=0.7)
    ax_density.minorticks_on()
    ax_density.legend(
        loc='upper right', fontsize=DENSITY_FONT_SIZES['legend'],
        handlelength=2.3, labelspacing=0.45, frameon=False
    )

    suffix = '_positive_r2_diagnostic' if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2 else ''
    figure_dir = helper_dir.parents[1] / 'figs' / suite
    figure_dir.mkdir(parents=True, exist_ok=True)
    for figure, stem in (
        (interval_fig, figure_dir / f'axial_radius_intervals{suffix}'),
        (density_fig, figure_dir / f'axial_radius_densities{suffix}'),
    ):
        figure.savefig(stem.with_suffix('.pdf'), dpi=600, bbox_inches='tight',
                       pad_inches=0.03, facecolor='white')
    # Tables are not figures: keep CSV summaries out of figs/.
    table_dir = helper_dir.parents[1] / 'tables' / suite
    table_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(table_dir / f'axial_radius_comparison{suffix}.csv')
    display(interval_fig)
    display(density_fig)
    plt.close(interval_fig)
    plt.close(density_fig)
    return interval_fig, density_fig


def plot_equivalent_ma_results(suite, distributions, summary):
    labels = list(distributions)
    interval_fig = plot_interval_summary(
        summary, labels, r'Equivalent dipole $M_A$ [GeV]'
    )
    ma_plot_range = (0.0, 2.0)
    linear_edges = np.linspace(*ma_plot_range, 181)
    density_fig, ax_density = plt.subplots(figsize=DENSITY_FIGSIZE,
                                           constrained_layout=True)
    peak = 0.0
    for (label, samples), color, style in zip(
        distributions.items(), INTERVAL_COLORS, ROW_STYLES
    ):
        # Normalized to the total sample, so that mass outside the plotted
        # M_A range is not renormalized away.
        counts, edges = np.histogram(
            samples, bins=linear_edges,
            weights=np.full(len(samples), 1.0 / len(samples)),
        )
        centers = (edges[1:] + edges[:-1]) / 2
        kernel_x = np.arange(-8, 9)
        kernel = np.exp(-0.5 * (kernel_x / 2.0)**2)
        kernel /= kernel.sum()
        density = counts / np.diff(edges)
        density = np.convolve(density, kernel, mode='same')
        peak = max(peak, density.max())
        ax_density.plot(centers, density, color=color, linewidth=2, label=label,
                        linestyle=style['dash'])

    # Headroom for the legend, which sits inside the axes.
    ax_density.set_ylim(0, LEGEND_HEADROOM * peak)
    ax_density.set_xlabel(r'Equivalent dipole $M_A$ [GeV]',
                          fontsize=DENSITY_FONT_SIZES['label'])
    ax_density.set_ylabel('Probability density',
                          fontsize=DENSITY_FONT_SIZES['label'])
    ax_density.set_xlim(ma_plot_range)
    ax_density.tick_params(labelsize=DENSITY_FONT_SIZES['tick'])
    ax_density.grid(color='#9AA4B2', alpha=0.18, linewidth=0.7)
    ax_density.minorticks_on()
    ax_density.legend(
        loc='upper right', fontsize=DENSITY_FONT_SIZES['legend'],
        handlelength=2.3, labelspacing=0.45, frameon=False
    )

    suffix = '_positive_r2_diagnostic' if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2 else ''
    figure_dir = helper_dir.parents[1] / 'figs' / suite
    for figure, stem in (
        (interval_fig, figure_dir / f'equivalent_ma_intervals{suffix}'),
        (density_fig, figure_dir / f'equivalent_ma_densities{suffix}'),
    ):
        figure.savefig(stem.with_suffix('.pdf'), dpi=600, bbox_inches='tight',
                       pad_inches=0.03, facecolor='white')
    table_dir = helper_dir.parents[1] / 'tables' / suite
    table_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(table_dir / f'equivalent_ma_comparison{suffix}.csv')
    display(interval_fig)
    display(density_fig)
    plt.close(interval_fig)
    plt.close(density_fig)
    return interval_fig, density_fig


In [ ]:
suite_outputs = {}
for suite, suite_label in SUITES:
    print(f'\n=== {suite_label}: {suite} ===')
    suite_result = build_suite_results(suite)
    suite_result['figures'] = plot_suite_results(
        suite, suite_result['distributions'], suite_result['summary'],
    )
    suite_result['equivalent_ma_figures'] = plot_equivalent_ma_results(
        suite, suite_result['equivalent_ma_distributions'],
        suite_result['equivalent_ma_summary'],
    )
    suite_outputs[suite] = suite_result
